# UC-WFS-1 — GetCapabilities and GetFeature via OGC WFS 2.0

**Persona:** Desktop GIS User

**Goal:** Ingest a set of point features via OGC Features POST, then use the
WFS 2.0 interface to issue `GetCapabilities` and `GetFeature` KVP requests
against the catalog, demonstrating interoperability with legacy WFS clients.

**Prerequisites:**
- GeoID platform running at `DYNASTORE_BASE_URL` (default `http://localhost:8080`)
- `wfs` and `features` extensions enabled in the active SCOPE
- Write-scoped JWT in `DYNASTORE_TOKEN` (or IDP client credentials set)

**References:**
- OGC WFS 2.0 (ISO 19142:2010)
- Routes: `GET /wfs/{catalog_id}?SERVICE=WFS&REQUEST=GetCapabilities`
  and `GET /wfs/{catalog_id}?SERVICE=WFS&REQUEST=GetFeature&TYPENAME={col}`

In [ ]:
import json
import os

import httpx
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True), override=False)

BASE_URL = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")

# Auto-provision token via IDP client_credentials when not already set
if not os.environ.get("DYNASTORE_TOKEN"):
    _idp = (os.environ.get("IDP_PUBLIC_URL") or os.environ.get("IDP_ISSUER_URL", "")).rstrip("/")
    _cid = os.environ.get("IDP_CLIENT_ID", "")
    _sec = os.environ.get("IDP_CLIENT_SECRET", "")
    if _idp and _cid and _sec:
        try:
            _r = httpx.post(
                f"{_idp}/protocol/openid-connect/token",
                data={"grant_type": "client_credentials", "client_id": _cid, "client_secret": _sec},
                timeout=10,
            )
            if _r.status_code == 200:
                os.environ["DYNASTORE_TOKEN"] = _r.json().get("access_token", "")
        except Exception:
            pass

TOKEN = (
    os.environ.get("DYNASTORE_TOKEN")
    or os.environ.get("DYNASTORE_WRITE_TOKEN")
    or os.environ.get("DYNASTORE_ADMIN_TOKEN")
    or ""
)

CATALOG_ID    = os.environ.get("CATALOG_ID", "demo-wfs")
COLLECTION_ID = os.environ.get("COLLECTION_ID", "cities")

headers = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}
client  = httpx.Client(base_url=BASE_URL, headers=headers, timeout=60.0)

print(f"Platform  : {BASE_URL}")
print(f"Catalog   : {CATALOG_ID}")
print(f"Collection: {COLLECTION_ID}")

## Step 1 — Create catalog and ingest point features

We create a demo catalog and a `cities` collection, then POST point features
for three Italian cities. The WFS interface will serve these features as GML
FeatureCollection responses.

In [ ]:
# --- catalog ---
r = client.post(
    "/features/catalogs",
    content=json.dumps({"id": CATALOG_ID, "title": "WFS demo catalog"}),
)
print(f"catalog   : {r.status_code}", "(already exists)" if r.status_code == 409 else "")

# --- collection ---
r = client.post(
    f"/features/catalogs/{CATALOG_ID}/collections",
    content=json.dumps({"id": COLLECTION_ID, "title": "Italian cities demo"}),
)
print(f"collection: {r.status_code}", "(already exists)" if r.status_code == 409 else "")

# --- items ---
cities = [
    {"id": "rome",   "name": "Rome",   "lon": 12.4964, "lat": 41.9028, "population": 2873000},
    {"id": "milan",  "name": "Milan",  "lon": 9.1900,  "lat": 45.4654, "population": 1352000},
    {"id": "naples", "name": "Naples", "lon": 14.2681, "lat": 40.8518, "population": 962589},
]

for city in cities:
    feat = {
        "type": "Feature",
        "id": city["id"],
        "geometry": {"type": "Point", "coordinates": [city["lon"], city["lat"]]},
        "properties": {"name": city["name"], "population": city["population"]},
    }
    r = client.post(
        f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
        content=json.dumps(feat),
    )
    if r.status_code in (200, 201):
        print(f"  item '{city['id']}': created")
    elif r.status_code == 409:
        print(f"  item '{city['id']}': already exists")
    else:
        print(f"  item '{city['id']}': {r.status_code} {r.text[:200]}")

## Step 2 — WFS GetCapabilities

`GET /wfs/{catalog_id}?SERVICE=WFS&REQUEST=GetCapabilities` returns an XML
WFS Capabilities document listing available feature types. We check the
response status and confirm the collection name appears in the XML.

In [ ]:
wfs_client = httpx.Client(
    base_url=BASE_URL,
    headers={"Authorization": f"Bearer {TOKEN}"},
    timeout=60.0,
)

r = wfs_client.get(
    f"/wfs/{CATALOG_ID}",
    params={"SERVICE": "WFS", "VERSION": "2.0.0", "REQUEST": "GetCapabilities"},
)
print(f"GetCapabilities status: {r.status_code}")

_wfs_active = r.status_code == 200

if _wfs_active:
    ct = r.headers.get("content-type", "")
    print(f"content-type : {ct}")
    body_text = r.text
    has_collection = COLLECTION_ID in body_text
    print(f"collection '{COLLECTION_ID}' in response: {has_collection}")
    # Print a short excerpt of the Capabilities XML
    print(body_text[:600])
elif r.status_code == 404:
    print("SKIP: /wfs endpoint not found — wfs extension may not be active.")
else:
    print(r.text[:300])

## Step 3 — WFS GetFeature

`GET /wfs/{catalog_id}?SERVICE=WFS&REQUEST=GetFeature&TYPENAME={collection}&COUNT=10`
returns a GML (or GeoJSON, with `OUTPUTFORMAT=application/json`) FeatureCollection
of the ingested features. We request GeoJSON output for readable results.

In [ ]:
if not _wfs_active:
    print("SKIP: WFS extension not active.")
else:
    r = wfs_client.get(
        f"/wfs/{CATALOG_ID}",
        params={
            "SERVICE": "WFS",
            "VERSION": "2.0.0",
            "REQUEST": "GetFeature",
            "TYPENAME": COLLECTION_ID,
            "COUNT": "10",
            "OUTPUTFORMAT": "application/json",
        },
    )
    print(f"GetFeature status: {r.status_code}")

    if r.status_code == 200:
        ct = r.headers.get("content-type", "")
        print(f"content-type : {ct}")
        if "json" in ct:
            fc = r.json()
            feats = fc.get("features", [])
            print(f"Features returned: {len(feats)}")
            for feat in feats:
                props = feat.get("properties", {})
                geom = feat.get("geometry", {})
                print(
                    f"  id={feat.get('id')}  name={props.get('name')}  "
                    f"coords={geom.get('coordinates') if geom else None}"
                )
        else:
            # GML response
            print(r.text[:600])
    elif r.status_code == 404:
        print("  404 — catalog not found or typename unknown.")
    else:
        print(r.text[:300])

## Step 4 — WFS GetFeature with BBOX filter

The WFS `BBOX` parameter filters features to those intersecting the specified
bounding box. We request only features within central Italy (11.0,40.0,14.0,43.5).

In [ ]:
if not _wfs_active:
    print("SKIP: WFS extension not active.")
else:
    r = wfs_client.get(
        f"/wfs/{CATALOG_ID}",
        params={
            "SERVICE": "WFS",
            "VERSION": "2.0.0",
            "REQUEST": "GetFeature",
            "TYPENAME": COLLECTION_ID,
            "BBOX": "11.0,40.0,14.0,43.5",
            "OUTPUTFORMAT": "application/json",
        },
    )
    print(f"GetFeature BBOX status: {r.status_code}")

    if r.status_code == 200:
        ct = r.headers.get("content-type", "")
        if "json" in ct:
            fc = r.json()
            feats = fc.get("features", [])
            print(f"Features in bbox: {len(feats)}")
            for feat in feats:
                print(f"  {feat.get('id')} — {feat.get('properties', {}).get('name')}")
        else:
            print(r.text[:400])
    else:
        print(r.text[:300])

    wfs_client.close()

## Teardown

In [ ]:
_TEARDOWN = os.environ.get("WFS_TEARDOWN", "true").lower() == "true"

if _TEARDOWN:
    r = client.delete(f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}")
    print(f"DELETE collection: {r.status_code}")
    r = client.delete(f"/features/catalogs/{CATALOG_ID}")
    print(f"DELETE catalog   : {r.status_code}")
else:
    print("Teardown skipped (set WFS_TEARDOWN=true to enable).")

## Result — WFS GetCapabilities URL

The WFS extension does not ship a dedicated web page. The canonical OGC
entry point is the GetCapabilities URL below.

In [ ]:
client.close()
print(
    f"WFS GetCapabilities URL:\n"
    f"  {BASE_URL}/wfs/{CATALOG_ID}?SERVICE=WFS&VERSION=2.0.0&REQUEST=GetCapabilities"
)